# Driving the 5G NR LDPC encoder from PYNQ

Drives the `ldpc_encoder` IP over a Xilinx AXI DMA, following
*PYNQ DMA Part 2* (https://discuss.pynq.io/t/tutorial-pynq-dma-part-2-using-the-dma-from-pynq/3134).

**Dataflow**

```
PS  --M_AXI_GP-->  s_axil             (config / status CSRs)
PS  --MM2S------>  AXI DMA --> s_axis  (code block in: 32-bit words, LSB-first)
m_axis --> AXI DMA --S2MM-->  PS       (codeword out: TLAST-terminated)
```

**Register map** (32-bit, byte addresses -- see `rtl/ldpc_encoder.v`)

| Addr | Name | Access | Fields |
|------|------|--------|--------|
| 0x0 | STATUS | RO | bit0 = `s_axis_tready`, bit1 = core busy |
| 0x4 | CONFIG | RW | bit0 = base graph (0=BG1, 1=BG2), bits[9:1] = lifting size Zc |
| 0x8 | IN_BITS | RW | bits[15:0] = code-block length = KB*Zc |
| 0xC | OUT_BITS | RW | bits[15:0] = codeword length = NB*Zc |

**Frame geometry** (from `rtl/ldpc_pkg.sv`): BG1 -> KB=22, NB=68; BG2 -> KB=10, NB=52.

> Two checks are provided: a quick **systematic passthrough** self-check (first KB*Zc
> output bits == info bits) and a full **golden-model** check that also verifies the
> parity bits against the 3GPP reference. For an over-the-air 5G codeword, drop the
> first 2*Zc punctured systematic bits.


In [ ]:
import numpy as np
from pynq import Overlay, allocate

# Point this at the .bit you generated. The matching .hwh must sit beside it,
# same basename (Vivado writes it under
# <proj>.gen/sources_1/bd/<bd>/hw_handoff/<bd>.hwh).
BITSTREAM = 'ldpc_encoder.bit'

ol = Overlay(BITSTREAM)

print('IPs in this overlay:')
for name in ol.ip_dict:
    print('   ', name)

## Bind the DMA and the LDPC IP

Set the handles to the names printed above. In a stock design they are usually
`axi_dma_0` and `ldpc_encoder_0`.


In [ ]:
dma  = ol.axi_dma_0        # Xilinx AXI Direct Memory Access
ldpc = ol.ldpc_encoder_0   # our custom AXI4-Lite CSR slave

assert hasattr(dma, 'sendchannel'), 'AXI DMA MM2S (Read channel) not enabled'
assert hasattr(dma, 'recvchannel'), 'AXI DMA S2MM (Write channel) not enabled'

## Registers, framing, bit packing, and hex helpers


In [ ]:
# CSR byte offsets, per rtl/ldpc_encoder.v
STATUS, CONFIG, IN_BITS, OUT_BITS = 0x0, 0x4, 0x8, 0xC

# Base-graph geometry, per rtl/ldpc_pkg.sv (KB = systematic cols, NB = total cols)
BG = {
    0: dict(name='BG1', KB=22, NB=68),
    1: dict(name='BG2', KB=10, NB=52),
}
ZC_MAX = 384

def frame_params(base_graph, zc):
    assert base_graph in (0, 1)
    assert 2 <= zc <= ZC_MAX
    b = BG[base_graph]
    in_bits, out_bits = b['KB'] * zc, b['NB'] * zc
    assert in_bits <= 0xFFFF and out_bits <= 0xFFFF, 'IN/OUT_BITS are 16-bit CSRs'
    return dict(name=b['name'], KB=b['KB'], NB=b['NB'], zc=zc,
                in_bits=in_bits, out_bits=out_bits,
                n_in_words=(in_bits + 31) // 32,
                n_out_words=(out_bits + 31) // 32)

def pack_bits_to_words(bits):
    '''LSB-first: bits[0] -> bit0 of word0. Zero-padded to whole 32-bit words.'''
    bits = np.asarray(bits, dtype=np.uint8) & 1
    nwords = (bits.size + 31) // 32
    padded = np.zeros(nwords * 32, dtype=np.uint8)
    padded[:bits.size] = bits
    w = padded.reshape(nwords, 32).astype(np.uint64)
    weights = np.uint64(1) << np.arange(32, dtype=np.uint64)
    return (w * weights).sum(axis=1).astype(np.uint32)

def unpack_words_to_bits(words, nbits):
    '''Inverse of pack_bits_to_words; returns exactly nbits bits (LSB-first).'''
    words = np.asarray(words, dtype=np.uint32)
    bits = ((words[:, None] >> np.arange(32, dtype=np.uint32)) & 1).astype(np.uint8)
    return bits.reshape(-1)[:nbits]

def col_to_hex(bits):
    '''LSB-first bit array -> hex string (MSB on the left), nibble-padded.'''
    bits = np.asarray(bits, dtype=np.uint8) & 1
    val = 0
    for j in range(bits.size):
        val |= int(bits[j]) << j
    width = max(1, (bits.size + 3) // 4)
    return f'0x{val:0{width}x}'

def cols_hex(bits, group):
    '''Split LSB-first bits into `group`-bit columns; return list of hex strings.'''
    bits = np.asarray(bits, dtype=np.uint8) & 1
    return [col_to_hex(bits[i:i + group]) for i in range(0, bits.size, group)]

def words_hex(words, n=8):
    '''First n DMA words as 0x%08x (what actually crosses the bus).'''
    return [f'0x{int(w):08x}' for w in np.asarray(words)[:n]]

def read_status():
    s = int(ldpc.read(STATUS))
    return dict(raw=hex(s), tready=bool(s & 1), busy=bool((s >> 1) & 1))

## Encode one frame

Order matters: program the CSRs **before** streaming -- the input buffer latches a
frame's configuration on its first AXIS beat. We arm S2MM (receive) before MM2S
(send) so the sink is ready the moment the core emits the codeword. The systematic
self-check is reported but **does not raise**, so you always get the data back.


In [ ]:
def encode_frame(base_graph, zc, info_bits=None, verbose=True):
    fp = frame_params(base_graph, zc)

    if info_bits is None:
        info_bits = np.random.randint(0, 2, size=fp['in_bits']).astype(np.uint8)
    info_bits = np.asarray(info_bits, dtype=np.uint8) & 1
    assert info_bits.size == fp['in_bits'], \
        f"need {fp['in_bits']} info bits, got {info_bits.size}"

    # 1) Configure (latched on the first input beat)
    ldpc.write(CONFIG,  (zc << 1) | (base_graph & 1))
    ldpc.write(IN_BITS,  fp['in_bits'])
    ldpc.write(OUT_BITS, fp['out_bits'])

    # 2) Contiguous DMA buffers
    in_buf  = allocate(shape=(fp['n_in_words'],),  dtype=np.uint32)
    out_buf = allocate(shape=(fp['n_out_words'],), dtype=np.uint32)
    in_buf[:] = pack_bits_to_words(info_bits)

    # 3) Arm receive, push send, wait for both
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    # 4) Unpack the full codeword (systematic part first); keep raw words too
    codeword = unpack_words_to_bits(np.asarray(out_buf), fp['out_bits'])
    raw_out  = np.asarray(out_buf).copy()
    in_buf.freebuffer()
    out_buf.freebuffer()

    ok = np.array_equal(codeword[:fp['in_bits']], info_bits)
    if verbose:
        print(f"[{'OK ' if ok else 'FAIL'}] {fp['name']} Zc={zc:3d}  "
              f"K={fp['in_bits']:5d}  N={fp['out_bits']:5d}  "
              f"systematic passthrough {'matches' if ok else 'MISMATCH'}")

    return dict(info=info_bits, codeword=codeword, raw_out=raw_out, fp=fp, ok=ok)

## Encode and compare in hex

Dumps the info and codeword in hex two ways: **flat** (the whole vector as one
LSB-first integer) and **per-column** (each Zc-bit column), so a bit-reversal or a
wrong-Zc shift is obvious at a glance.


In [ ]:
np.random.seed(0)
r = encode_frame(0, 64)                 # BG1, Zc=64 (no longer raises on mismatch)
info, cw, fp = r['info'], r['codeword'], r['fp']
zc, KB, NB = fp['zc'], fp['KB'], fp['NB']

print('status:', read_status())
print()

print('info          (flat hex):', col_to_hex(info))
print('codeword[sys] (flat hex):', col_to_hex(cw[:fp['in_bits']]))
print('codeword[all] (flat hex):', col_to_hex(cw))
print()

info_cols, cw_cols = cols_hex(info, zc), cols_hex(cw, zc)
ndiff = 0
for c in range(NB):
    tag = 'sys' if c < KB else 'par'
    if c < KB:
        same = (info_cols[c] == cw_cols[c])
        ndiff += (not same)
        print(f'col {c:>3} [{tag}] {"ok  " if same else "DIFF"}  info={info_cols[c]}  cw={cw_cols[c]}')
    else:
        print(f'col {c:>3} [{tag}]        cw={cw_cols[c]}')

print()
print(f'systematic columns differing: {ndiff}/{KB}')
d = np.flatnonzero(cw[:fp['in_bits']] != info)
print('first differing bit index   :', int(d[0]) if d.size else None)
print()
print('first 8 input  DMA words:', words_hex(pack_bits_to_words(info)))
print('first 8 output DMA words:', words_hex(r['raw_out']))

## Full verification against the golden model

This checks the **whole codeword, parity included**, against the 3GPP reference --
the same `LdpcEncoderGoldenModel` the RTL simulation uses. The expected codeword is
exactly `info ++ encode(info, Zc, bg_idx=base_graph+1)` (mirroring `sim/pyuvm_tb/bfm.py`).

`golden_model.py` and the CSR `mem/` files are **bundled in this folder**, so it runs
as-is once you copy the whole `pynq/` folder to the board. (If you keep them
elsewhere, adjust `GOLDEN_DIR` / `MEM_DIR`.)


In [ ]:
import sys

GOLDEN_DIR = '.'      # directory containing golden_model.py
MEM_DIR    = 'mem'    # directory with col_indices / row_ptr / values* / row_schedule .mem

if GOLDEN_DIR not in sys.path:
    sys.path.insert(0, GOLDEN_DIR)
from golden_model import LdpcEncoderGoldenModel

gm = LdpcEncoderGoldenModel()
gm.load_csr_data(MEM_DIR)
print('golden model loaded from', repr(MEM_DIR))

def golden_codeword(base_graph, zc, info_bits):
    '''Full expected codeword (LSB-first, column-major) = info ++ parity, in the
    order the RTL streams it. Mirrors sim/pyuvm_tb/bfm.py exactly.'''
    info = np.asarray(info_bits, dtype=np.uint8) & 1
    parity = gm.encode(info.tolist(), int(zc), bg_idx=base_graph + 1, version='3gpp')
    return np.concatenate([info, np.asarray(parity, dtype=np.uint8)])

def verify_frame(base_graph, zc, info_bits=None, show=8, verbose=True):
    res = encode_frame(base_graph, zc, info_bits=info_bits, verbose=False)
    fp, cw = res['fp'], res['codeword']
    exp = golden_codeword(base_graph, zc, res['info'])
    assert exp.size == cw.size == fp['out_bits'], 'length mismatch vs golden'

    s = fp['in_bits']
    npar = fp['out_bits'] - s
    sys_bad = int(np.count_nonzero(cw[:s] != exp[:s]))
    par_bad = int(np.count_nonzero(cw[s:] != exp[s:]))
    res['ok'] = ok = (sys_bad == 0 and par_bad == 0)
    if verbose:
        print(f"[{'OK ' if ok else 'FAIL'}] {fp['name']} Zc={zc:3d}  "
              f"sys {s - sys_bad}/{s} ok, par {npar - par_bad}/{npar} ok  "
              f"(sys_bad={sys_bad}, par_bad={par_bad})")

    if show:
        zc_, KB, NB = fp['zc'], fp['KB'], fp['NB']
        hw, go = cols_hex(cw, zc_), cols_hex(exp, zc_)
        print('  parity columns (golden vs hw):')
        for c in range(KB, min(KB + show, NB)):
            print(f'    col {c:>3} [{"ok  " if go[c] == hw[c] else "DIFF"}] '
                  f'golden={go[c]}  hw={hw[c]}')
    return dict(res=res, expected=exp, sys_bad=sys_bad, par_bad=par_bad, ok=ok)

np.random.seed(0)
_ = verify_frame(0, 64)     # BG1, Zc=64: full codeword vs 3GPP golden model

## Localize the corruption

Targeted probes for a systematic mismatch. In ZC_SMALL the core writes
**bit-identical data to all four physical sub-banks** (`data_segment[0..3]` are
equal), so any *per-sub-bank* difference cannot come from the data/logic path --
it points at the memory itself (timing / placement). These probes confirm the
shape and tell timing from logic:

- **(a) all-zeros / all-ones** -- data-independent; isolates stuck/extra bits.
- **(b) per sub-bank** (ZC_SMALL: sub-bank = column % 4) -- which `ram[i]` is bad.
- **(c) stability** -- same input 10x. Identical => deterministic; varies =>
  timing / metastability.
- **(d) ZC_LARGE** -- one column spans all four sub-banks (pf=1).

Also open the **implemented timing report** (`report_timing_summary`): a negative
WNS on the `codeword_generator` BRAM paths is the prime suspect.


In [ ]:
KBZ = frame_params(0, 64)['in_bits']

def sys_bad(res):
    '''Count systematic bits that differ from the info sent.'''
    return int(np.count_nonzero(res['codeword'][:res['fp']['in_bits']] != res['info']))

# (a) data-independent patterns
for val, name in [(0, 'all-zeros'), (1, 'all-ones')]:
    res = encode_frame(0, 64, info_bits=np.full(KBZ, val, np.uint8), verbose=False)
    print(f'{name:9s}: {sys_bad(res):4d} systematic bits wrong')

# (b) per physical sub-bank (ZC_SMALL: sub-bank = column % 4)
res = encode_frame(0, 64, verbose=False)
zc, KB = res['fp']['zc'], res['fp']['KB']
ic, cc = cols_hex(res['info'], zc), cols_hex(res['codeword'], zc)
for sb in range(4):
    cols = [c for c in range(KB) if c % 4 == sb]
    print(f'sub-bank {sb}: {sum(ic[c] != cc[c] for c in cols)}/{len(cols)} columns wrong')

# (c) stability: same input 10x
seed = np.random.RandomState(1).randint(0, 2, KBZ).astype(np.uint8)
sigs = {encode_frame(0, 64, info_bits=seed, verbose=False)['codeword'].tobytes()
        for _ in range(10)}
print('10 identical runs ->', len(sigs), 'distinct codeword(s):',
      'DETERMINISTIC' if len(sigs) == 1 else 'NON-deterministic (timing/metastability)')

# (d) cross-check a ZC_LARGE frame (one column spans all four sub-banks)
print('ZC_LARGE (BG2 Zc=256):', sys_bad(encode_frame(1, 256, verbose=False)), 'sys bits wrong')

## Sweep lifting sizes (full golden-model check)

Exercises all three Zc groups (<=96, <=192, >192); each frame is fully verified
(systematic **and** parity) against the golden model.


In [ ]:
sweeps = [
    (0, [4, 16, 40, 64, 96]),    # BG1, ZC_SMALL up to the boundary
    (0, [104, 144, 192]),        # BG1, ZC_MEDIUM
    (1, [208, 256, 320, 384]),   # BG2, ZC_LARGE
]

fails = 0
for bg, zcs in sweeps:
    for zc in zcs:
        if not verify_frame(bg, zc, show=0)['ok']:
            fails += 1
print()
print('sweep done:', 'all passed' if fails == 0 else f'{fails} failure(s)')

## Troubleshooting

- **Systematic passthrough MISMATCH** -- read the per-column / per-sub-bank output:
  - *one sub-bank clean, others sporadic bit flips* -> **timing closure** on the
    codeword RAM (identical data is written to all four banks, so a per-bank
    difference is physical, not logic). Check `report_timing_summary` (WNS) and the
    PL clock; lower FCLK or add a pipeline stage.
  - *columns unrelated to info* -> bit-packing / IP-DMA wiring, or CONFIG not latched
    (read back `hex(ldpc.read(0x4))`; expect `(zc<<1)|bg`).
  - *columns match but bit-reversed* -> flip the bit order in the pack/unpack helpers.
  - *codeword columns look like info shifted* -> wrong Zc grouping (CONFIG Zc differs).
  - *output words all zero* -> core never ran: check `clk_i`, active-low `arst_ni`.
- **Systematic OK but parity DIFF** -> a real encode bug (or wrong base graph / Zc);
  compare the per-column golden-vs-hw parity printed by `verify_frame`.
- **`sendchannel.wait()` hangs** -- MM2S->`s_axis`, and `IN_BITS` == bits packed.
- **`recvchannel.wait()` hangs** -- `m_axis`->S2MM, `m_axis_tlast`, `out_buf` >= NB*Zc.
- **`ModuleNotFoundError: golden_model`** -- copy the whole `pynq/` folder (it bundles
  `golden_model.py` and `mem/`), or set `GOLDEN_DIR` / `MEM_DIR`.
